# Add Shot Context Labels To Origin-Shot Chains

This notebook adds context labels to the origin-shot chain dataset.

The goal is to support later context-stratified analysis of 5v5 outside-origin shots.

Context labels created here are descriptive proxy labels, not final causal claims.

Primary context families:

- offensive-zone faceoff within 5 seconds
- offensive-zone entry proxy within 5 seconds
- settled offensive-zone possession proxy
- other / unknown

The 5-second windows intentionally follow a common hockey analytics convention: evaluate plays shortly after the puck enters the offensive zone or shortly after an offensive-zone faceoff.

Because this dataset does not provide continuous puck-crossing events at the blue line, the entry label is called an **entry proxy**, not a true rush label.

## 1. Setup

Load libraries, define paths, and configure display options.

In [1]:
# Import libraries used for context-label construction

from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
# Define project paths for raw and processed data

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_RAW = PROJECT_ROOT / "data" / "raw" / "halo_2026"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Raw data exists:", DATA_RAW.exists())
print("Processed data exists:", DATA_PROCESSED.exists())

Project root: c:\Users\rinal\hockey-analytics\outside-shot-value
Raw data exists: True
Processed data exists: True


In [3]:
# Configure pandas display options for readable notebook tables

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:0.6f}")

## 2. Load Inputs

Load the labeled origin-shot chain dataset from Notebook 06 and the raw event table.

Notebook 08 uses the raw event table to recover origin-shot coordinates and to look backward within the same game, period, and sequence for context clues.

In [4]:
# Load processed chains and raw events

chains = pd.read_parquet(DATA_PROCESSED / "origin_shot_sequences_labeled.parquet")
events = pd.read_parquet(DATA_RAW / "events.parquet")

print("chains:", chains.shape)
print("events:", events.shape)

chains: (48673, 75)
events: (1800464, 24)


In [5]:
# Inspect columns needed for context labeling

print("Chain columns:")
display(chains.columns.tolist())

print("Event columns:")
display(events.columns.tolist())

Chain columns:


['chain_id',
 'game_id',
 'period',
 'sequence_id',
 'team_id',
 'raw_first_chance_team_id',
 'n_team_id_overrides_for_chain',
 'chain_start_time',
 'chain_end_time',
 'chain_duration_seconds',
 'n_chances_in_chain',
 'anchor_origin_event_id',
 'anchor_origin_period_time',
 'anchor_origin_location',
 'anchor_first_chance_event_id',
 'anchor_first_chance_time',
 'anchor_first_event_type',
 'anchor_first_chance_location',
 'first_evaluated_chance_event_id',
 'first_evaluated_chance_time',
 'first_evaluated_event_type',
 'first_evaluated_location',
 'first_evaluated_xg',
 'chain_max_xg',
 'chain_sum_xg',
 'has_deflection',
 'n_deflections',
 'first_deflection_event_id',
 'first_deflection_time',
 'deflection_max_xg',
 'deflection_sum_xg',
 'has_followup_chance',
 'n_followup_chances',
 'first_followup_event_id',
 'first_followup_time',
 'followup_max_xg',
 'followup_sum_xg',
 'chain_goal',
 'chain_goal_event_id',
 'chain_goal_time',
 'chain_goal_player_id',
 'chain_goal_player_name',
 'ch

Event columns:


['game_id',
 'period',
 'period_time',
 'game_stint',
 'sl_event_id',
 'sequence_id',
 'player_id',
 'player_name',
 'team',
 'team_id',
 'opp_team',
 'opp_team_id',
 'event_type',
 'outcome',
 'flags',
 'description',
 'detail',
 'sl_xg_all_shots',
 'x',
 'y',
 'x_adj',
 'y_adj',
 'has_tracking_data',
 'event_player_tracked']

## 3. Validate Required Inputs

Validate that the notebook has the columns required to build context labels.

The context logic depends on:

- one row per chain
- an origin event id
- chain team identity
- game/period/sequence timing
- raw event timing and coordinates

In [6]:
# Validate required columns before building context labels

required_chain_columns = [
    "chain_id",
    "game_id",
    "period",
    "sequence_id",
    "team_id",
    "team_side",
    "home_team",
    "away_team",
    "home_team_id",
    "away_team_id",
    "anchor_origin_event_id",
    "anchor_origin_period_time",
    "anchor_origin_location",
    "is_5v5",
    "origin_tracking_available",
    "origin_tracking_error_flag",
    "observed_origin_attackers_in_slot",
    "observed_origin_defenders_in_slot",
]

required_event_columns = [
    "game_id",
    "period",
    "sequence_id",
    "sl_event_id",
    "period_time",
    "event_type",
    "team",
    "x_adj",
    "y_adj",
]

missing_required_columns = {
    "chains": [c for c in required_chain_columns if c not in chains.columns],
    "events": [c for c in required_event_columns if c not in events.columns],
}

missing_required_columns

{'chains': [], 'events': []}

In [7]:
# Validate one-row-per-chain structure

input_validation = {
    "chain_rows": len(chains),
    "unique_chain_ids": chains["chain_id"].nunique(),
    "duplicate_chain_ids": chains.duplicated("chain_id").sum(),
    "missing_origin_event_id": chains["anchor_origin_event_id"].isna().sum(),
    "missing_origin_time": chains["anchor_origin_period_time"].isna().sum(),
}

input_validation

{'chain_rows': 48673,
 'unique_chain_ids': 48673,
 'duplicate_chain_ids': np.int64(0),
 'missing_origin_event_id': np.int64(2),
 'missing_origin_time': np.int64(2)}

## 4. Attach Origin Coordinates

Notebook 06 did not permanently save origin-shot coordinates.

Join raw event coordinates back onto the chain dataset using:

- `game_id`
- `anchor_origin_event_id`

These coordinates are needed to exclude behind-blue-line shots and to identify offensive-zone context.

In [8]:
# Build an origin-event coordinate lookup from the raw event table

origin_event_lookup_columns = [
    "game_id",
    "sl_event_id",
    "period_time",
    "event_type",
    "team",
    "x_adj",
    "y_adj",
]

origin_event_lookup = (
    events[origin_event_lookup_columns]
    .rename(
        columns={
            "sl_event_id": "anchor_origin_event_id",
            "period_time": "raw_anchor_origin_period_time",
            "event_type": "anchor_origin_event_type_raw",
            "team": "anchor_origin_event_team",
            "x_adj": "anchor_origin_x_adj",
            "y_adj": "anchor_origin_y_adj",
        }
    )
    .drop_duplicates(["game_id", "anchor_origin_event_id"])
)

origin_event_lookup.head()

,game_id,anchor_origin_event_id,raw_anchor_origin_period_time,anchor_origin_event_type_raw,anchor_origin_event_team,anchor_origin_x_adj,anchor_origin_y_adj
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc,0,0E-9,faceoff,NaN,NaN,NaN
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,0E-9,faceoff,GR,-0.201431,-0.755550
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc,2,0E-9,faceoff,CLE,0.201431,0.755550
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc,3,0.070000000,lpr,CLE,-0.807243,-1.257381
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc,4,0.130000000,pass,CLE,-0.304306,-0.845894


In [9]:
# Attach raw origin-event coordinates to the chain dataset

chains_with_coords = chains.merge(
    origin_event_lookup,
    on=["game_id", "anchor_origin_event_id"],
    how="left",
    validate="many_to_one",
)

coord_join_validation = {
    "rows_before": len(chains),
    "rows_after": len(chains_with_coords),
    "duplicate_chain_ids": chains_with_coords.duplicated("chain_id").sum(),
    "missing_anchor_origin_x_adj": chains_with_coords["anchor_origin_x_adj"].isna().sum(),
    "missing_anchor_origin_y_adj": chains_with_coords["anchor_origin_y_adj"].isna().sum(),
}

coord_join_validation

{'rows_before': 48673,
 'rows_after': 48673,
 'duplicate_chain_ids': np.int64(0),
 'missing_anchor_origin_x_adj': np.int64(2),
 'missing_anchor_origin_y_adj': np.int64(2)}

In [10]:
# Create a reusable offensive-zone origin flag

OFFENSIVE_BLUE_LINE_X_ADJ = 25

chains_with_coords["is_offensive_zone_origin"] = (
    chains_with_coords["anchor_origin_x_adj"] >= OFFENSIVE_BLUE_LINE_X_ADJ
)

chains_with_coords["is_behind_offensive_blue_line_origin"] = (
    chains_with_coords["anchor_origin_x_adj"].notna()
    & ~chains_with_coords["is_offensive_zone_origin"]
)

chains_with_coords[
    [
        "anchor_origin_x_adj",
        "anchor_origin_y_adj",
        "anchor_origin_location",
        "is_offensive_zone_origin",
        "is_behind_offensive_blue_line_origin",
    ]
].head()

,anchor_origin_x_adj,anchor_origin_y_adj,anchor_origin_location,is_offensive_zone_origin,is_behind_offensive_blue_line_origin
0,32.889107,25.397057,outside,True,False
1,52.107956,-8.799999,outside,True,False
2,64.681470,-20.870590,slot,True,False
3,77.757950,5.785294,slot,True,False
4,57.640305,26.405882,outside,True,False


## 5. Create Team Keys For Event Matching

The raw event table uses team names, while the processed chain table carries team ids and home/away metadata.

For context labeling, create a readable team key on the chain table that can match the raw event `team` field.

This lets us identify whether a prior event in the same sequence belongs to the same attacking team as the chain.

In [11]:
# Create the chain attacking-team name from home/away metadata

chains_with_coords["chain_team_name"] = np.select(
    [
        chains_with_coords["team_side"].eq("home"),
        chains_with_coords["team_side"].eq("away"),
    ],
    [
        chains_with_coords["home_team"],
        chains_with_coords["away_team"],
    ],
    default=pd.NA,
)

team_key_validation = {
    "rows": len(chains_with_coords),
    "missing_chain_team_name": chains_with_coords["chain_team_name"].isna().sum(),
    "team_side_counts": chains_with_coords["team_side"].value_counts(dropna=False).to_dict(),
}

team_key_validation

{'rows': 48673,
 'missing_chain_team_name': np.int64(0),
 'team_side_counts': {'home': 24836, 'away': 23837}}

In [12]:
# Inspect whether raw event team names align with chain team names

chain_team_names = set(chains_with_coords["chain_team_name"].dropna().unique())
event_team_names = set(events["team"].dropna().unique())

team_name_alignment = {
    "n_chain_team_names": len(chain_team_names),
    "n_event_team_names": len(event_team_names),
    "chain_names_not_in_events": sorted(chain_team_names - event_team_names)[:20],
    "event_names_not_in_chains": sorted(event_team_names - chain_team_names)[:20],
}

team_name_alignment

{'n_chain_team_names': 32,
 'n_event_team_names': 32,
 'chain_names_not_in_events': [],
 'event_names_not_in_chains': []}

## 6. Event Grammar For Context Candidates

Before labeling contexts, inspect the event types that could indicate:

- faceoffs
- entries / post-entry situations
- offensive-zone possession events

This keeps us from pretending the data has cleaner context labels than it actually does.

In [13]:
# Count event types relevant to context labeling

event_type_counts = (
    events["event_type"]
    .value_counts(dropna=False)
    .rename_axis("event_type")
    .reset_index(name="events")
)

event_type_counts.head(40)

,event_type,events
0,pass,402670
1,lpr,338836
2,reception,304903
3,carry,110362
4,failedpasslocation,97366
5,faceoff,84621
6,block,66233
7,puckprotection,54827
8,shot,53971
9,pressure,53961


In [14]:
# Inspect candidate context event types

candidate_event_types = [
    "faceoff",
    "controlledentry",
    "controlledentryagainst",
    "dumpin",
    "dumpinagainst",
    "carry",
    "pass",
    "reception",
    "shot",
    "goal",
    "whistle",
    "offside",
    "icing",
]

candidate_event_type_counts = event_type_counts[
    event_type_counts["event_type"].isin(candidate_event_types)
].reset_index(drop=True)

candidate_event_type_counts

,event_type,events
0,pass,402670
1,reception,304903
2,carry,110362
3,faceoff,84621
4,shot,53971
5,controlledentryagainst,38081
6,dumpin,34740
7,dumpinagainst,32182
8,whistle,25397
9,goal,5667


## 7. Prepare Event Context Table

Create a compact event table for context labeling.

For each raw event, we keep:

- game/period/sequence
- event id and time
- event type
- event team name
- adjusted coordinates

Later, when comparing an event to a chain, we convert the event's x-coordinate into the chain team's attacking direction.

In [15]:
# Prepare compact event table for context labeling

event_context = events[
    [
        "game_id",
        "period",
        "sequence_id",
        "sl_event_id",
        "period_time",
        "event_type",
        "team",
        "x_adj",
        "y_adj",
    ]
].copy()

event_context = event_context.rename(
    columns={
        "sl_event_id": "context_event_id",
        "period_time": "context_event_time",
        "event_type": "context_event_type",
        "team": "context_event_team",
        "x_adj": "context_event_x_adj_for_event_team",
        "y_adj": "context_event_y_adj_for_event_team",
    }
)

event_context["context_event_time"] = pd.to_numeric(
    event_context["context_event_time"],
    errors="coerce",
)

event_context.head()

,game_id,period,sequence_id,context_event_id,context_event_time,context_event_type,context_event_team,context_event_x_adj_for_event_team,context_event_y_adj_for_event_team
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,1,0,0.000000,faceoff,NaN,NaN,NaN
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,1,1,0.000000,faceoff,GR,-0.201431,-0.755550
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,1,2,0.000000,faceoff,CLE,0.201431,0.755550
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,1,3,0.070000,lpr,CLE,-0.807243,-1.257381
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,1,4,0.130000,pass,CLE,-0.304306,-0.845894


In [16]:
# Validate compact event context table

event_context_validation = {
    "rows": len(event_context),
    "missing_context_event_time": event_context["context_event_time"].isna().sum(),
    "missing_context_event_team": event_context["context_event_team"].isna().sum(),
    "missing_context_event_x_adj": event_context["context_event_x_adj_for_event_team"].isna().sum(),
    "duplicate_context_event_ids": event_context.duplicated(["game_id", "context_event_id"]).sum(),
}

event_context_validation

{'rows': 1800464,
 'missing_context_event_time': np.int64(0),
 'missing_context_event_team': np.int64(64310),
 'missing_context_event_x_adj': np.int64(64310),
 'duplicate_context_event_ids': np.int64(0)}

## 8. Add Offensive-Zone Faceoff Context

Identify the most recent faceoff before the origin shot within the same game, period, and sequence.

A chain is labeled as offensive-zone faceoff context when:

- the prior faceoff occurred within 5 seconds of the origin shot
- the faceoff was in the offensive zone for the chain team

This creates a parallel to the five-second post-entry convention.

In [17]:
# Build prior-faceoff lookup table

faceoff_events = event_context[
    event_context["context_event_type"].eq("faceoff")
].copy()

faceoff_events = faceoff_events.rename(
    columns={
        "context_event_id": "prior_faceoff_event_id",
        "context_event_time": "prior_faceoff_time",
        "context_event_team": "prior_faceoff_team",
        "context_event_x_adj_for_event_team": "prior_faceoff_x_adj_for_event_team",
        "context_event_y_adj_for_event_team": "prior_faceoff_y_adj_for_event_team",
    }
)

faceoff_events = faceoff_events[
    [
        "game_id",
        "period",
        "sequence_id",
        "prior_faceoff_event_id",
        "prior_faceoff_time",
        "prior_faceoff_team",
        "prior_faceoff_x_adj_for_event_team",
        "prior_faceoff_y_adj_for_event_team",
    ]
].sort_values(["game_id", "period", "sequence_id", "prior_faceoff_time"])

faceoff_events.head()

,game_id,period,sequence_id,prior_faceoff_event_id,prior_faceoff_time,prior_faceoff_team,prior_faceoff_x_adj_for_event_team,prior_faceoff_y_adj_for_event_team
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,1,0,0.000000,NaN,NaN,NaN
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,1,1,0.000000,GR,-0.201431,-0.755550
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,1,2,0.000000,CLE,0.201431,0.755550
77,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2,77,101.030000,NaN,NaN,NaN
78,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2,78,101.030000,CLE,-69.204450,-21.876802


In [18]:
# Prepare chains for prior-event asof joins

chains_for_context = chains_with_coords.copy()

chains_for_context["anchor_origin_period_time"] = pd.to_numeric(
    chains_for_context["anchor_origin_period_time"],
    errors="coerce",
)

chains_for_context = chains_for_context.sort_values(
    ["game_id", "period", "sequence_id", "anchor_origin_period_time"]
)

chains_for_context.head()

,chain_id,game_id,period,sequence_id,team_id,raw_first_chance_team_id,n_team_id_overrides_for_chain,chain_start_time,chain_end_time,chain_duration_seconds,n_chances_in_chain,anchor_origin_event_id,anchor_origin_period_time,anchor_origin_location,anchor_first_chance_event_id,anchor_first_chance_time,anchor_first_event_type,anchor_first_chance_location,first_evaluated_chance_event_id,first_evaluated_chance_time,first_evaluated_event_type,first_evaluated_location,first_evaluated_xg,chain_max_xg,chain_sum_xg,has_deflection,n_deflections,first_deflection_event_id,first_deflection_time,deflection_max_xg,deflection_sum_xg,has_followup_chance,n_followup_chances,first_followup_event_id,first_followup_time,followup_max_xg,followup_sum_xg,chain_goal,chain_goal_event_id,chain_goal_time,chain_goal_player_id,chain_goal_player_name,chain_any_goal_within_2s_flag,origin_tracking_rows,origin_identified_tracking_players,origin_missing_tracking_player_rows,origin_attacker_skaters_tracked,origin_defender_skaters_tracked,observed_origin_attackers_in_slot,observed_origin_defenders_in_slot,origin_goalies_tracked,origin_shooter_tracked,origin_tracking_available,origin_tracking_error_flag,origin_tracking_completeness_bucket,home_team,away_team,home_team_id,away_team_id,team_side,period_time_start,game_stint,period_time_end,n_home_skaters,n_away_skaters,is_home_net_empty,is_away_net_empty,home_score,away_score,origin_time_within_stint,is_5v5,team_n_skaters,opponent_n_skaters,team_net_empty,opponent_net_empty,raw_anchor_origin_period_time,anchor_origin_event_type_raw,anchor_origin_event_team,anchor_origin_x_adj,anchor_origin_y_adj,is_offensive_zone_origin,is_behind_offensive_blue_line_origin,chain_team_name
16,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s2_td7...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,0,125.270000,125.270000,0.000000,1,102,125.270000,slot,102,125.270000,shot,slot,102,125.270000,shot,slot,0.106327,0.106327,0.106327,False,0,<NA>,NaN,0.000000,0.000000,False,0,<NA>,NaN,0.000000,0.000000,False,<NA>,NaN,NaN,NaN,False,5,1,4,1,0,0,0,0,True,True,False,partial_observed,GR,CLE,6cac12e2-0546-2c1a-689f-ab26d8a6355a,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,away,115.030000,15.000000,168.030000,5.000000,5.000000,False,False,0.000000,0.000000,True,True,5.000000,5.000000,False,False,125.270000000,shot,CLE,67.089810,-1.760323,True,False,CLE
17,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s2_td7...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,0,142.200000,142.200000,0.000000,1,111,142.200000,outside,111,142.200000,shot,outside,111,142.200000,shot,outside,0.002135,0.002135,0.002135,False,0,<NA>,NaN,0.000000,0.000000,False,0,<NA>,NaN,0.000000,0.000000,False,<NA>,NaN,NaN,NaN,False,9,5,4,2,3,0,0,0,True,True,False,partial_observed,GR,CLE,6cac12e2-0546-2c1a-689f-ab26d8a6355a,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,away,115.030000,15.000000,168.030000,5.000000,5.000000,False,False,0.000000,0.000000,True,True,5.000000,5.000000,False,False,142.200000000,shot,CLE,37.919224,-26.907381,True,False,CLE
18,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s2_td7...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,0,155.270000,155.270000,0.000000,1,122,155.270000,slot,122,155.270000,shot,slot,122,155.270000,shot,slot,0.018751,0.018751,0.018751,False,0,<NA>,NaN,0.000000,0.000000,False,0,<NA>,NaN,0.000000,0.000000,False,<NA>,NaN,NaN,NaN,False,10,4,6,1,3,1,2,0,False,True,False,partial_observed,GR,CLE,6cac12e2-0546-2c1a-689f-ab26d8a6355a,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,away,115.030000,15.000000,168.030000,5.000000,5.000000,False,False,0.000000,0.000000,True,True,5.000000,5.000000,False,False,155.270000000,shot,CLE,79.663345,19.866150,True,False,CLE
19,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s2_td7...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2,d7ff41e7-8310-f2ab-3c79-6ad3d1f4f019,d7ff41e7-831

In [20]:
# Attach the most recent prior faceoff in the same game, period, and sequence

chains_for_context_valid_time = chains_for_context[
    chains_for_context["anchor_origin_period_time"].notna()
].copy()

chains_for_context_missing_time = chains_for_context[
    chains_for_context["anchor_origin_period_time"].isna()
].copy()

chains_for_context_valid_time["anchor_origin_period_time"] = pd.to_numeric(
    chains_for_context_valid_time["anchor_origin_period_time"],
    errors="coerce",
)

faceoff_events_for_asof = faceoff_events.copy()

faceoff_events_for_asof["prior_faceoff_time"] = pd.to_numeric(
    faceoff_events_for_asof["prior_faceoff_time"],
    errors="coerce",
)

faceoff_events_for_asof = faceoff_events_for_asof[
    faceoff_events_for_asof["prior_faceoff_time"].notna()
].copy()

# Pandas merge_asof requires the merge key to be globally sorted.
# Sort by time first, then by grouping keys.
chains_for_context_valid_time = chains_for_context_valid_time.sort_values(
    ["anchor_origin_period_time", "game_id", "period", "sequence_id"]
).reset_index(drop=True)

faceoff_events_for_asof = faceoff_events_for_asof.sort_values(
    ["prior_faceoff_time", "game_id", "period", "sequence_id"]
).reset_index(drop=True)

chains_with_faceoff_valid_time = pd.merge_asof(
    chains_for_context_valid_time,
    faceoff_events_for_asof,
    left_on="anchor_origin_period_time",
    right_on="prior_faceoff_time",
    by=["game_id", "period", "sequence_id"],
    direction="backward",
    allow_exact_matches=True,
)

# Reattach rows with missing origin time. They cannot have a valid prior faceoff-time match.
faceoff_added_columns = [
    "prior_faceoff_event_id",
    "prior_faceoff_time",
    "prior_faceoff_team",
    "prior_faceoff_x_adj_for_event_team",
    "prior_faceoff_y_adj_for_event_team",
]

for col in faceoff_added_columns:
    chains_for_context_missing_time[col] = pd.NA

chains_with_faceoff = pd.concat(
    [
        chains_with_faceoff_valid_time,
        chains_for_context_missing_time,
    ],
    ignore_index=True,
)

chains_with_faceoff["seconds_since_prior_faceoff"] = (
    chains_with_faceoff["anchor_origin_period_time"]
    - pd.to_numeric(chains_with_faceoff["prior_faceoff_time"], errors="coerce")
)

chains_with_faceoff = chains_with_faceoff.sort_values("chain_id").reset_index(drop=True)

chains_with_faceoff[
    [
        "chain_id",
        "anchor_origin_period_time",
        "prior_faceoff_time",
        "seconds_since_prior_faceoff",
        "chain_team_name",
        "prior_faceoff_team",
        "prior_faceoff_x_adj_for_event_team",
    ]
].head()

,chain_id,anchor_origin_period_time,prior_faceoff_time,seconds_since_prior_faceoff,chain_team_name,prior_faceoff_team,prior_faceoff_x_adj_for_event_team
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s11_td...,591.730000,589.030000,2.700000,CLE,CLE,69.104380
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,722.400000,620.030000,102.370000,GR,CLE,-19.413269
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,736.330000,620.030000,116.300000,GR,CLE,-19.413269
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,765.730000,620.030000,145.700000,GR,CLE,-19.413269
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s13_t6...,923.800000,773.030000,150.770000,GR,CLE,69.104380


In [21]:
# Validate faceoff asof join row preservation

faceoff_lookup_validation = {
    "rows": len(chains_with_faceoff),
    "unique_chain_ids": chains_with_faceoff["chain_id"].nunique(),
    "duplicate_chain_ids": chains_with_faceoff.duplicated("chain_id").sum(),
    "missing_origin_time": chains_with_faceoff["anchor_origin_period_time"].isna().sum(),
    "rows_with_prior_faceoff": chains_with_faceoff["prior_faceoff_event_id"].notna().sum(),
    "negative_seconds_since_prior_faceoff": int(
        (chains_with_faceoff["seconds_since_prior_faceoff"] < 0).sum()
    ),
}

faceoff_lookup_validation

{'rows': 48673,
 'unique_chain_ids': 48673,
 'duplicate_chain_ids': np.int64(0),
 'missing_origin_time': np.int64(2),
 'rows_with_prior_faceoff': np.int64(48652),
 'negative_seconds_since_prior_faceoff': 0}

In [22]:
# Convert prior faceoff coordinates into the chain team's attacking direction

same_team_faceoff = chains_with_faceoff["prior_faceoff_team"].eq(
    chains_with_faceoff["chain_team_name"]
)

chains_with_faceoff["prior_faceoff_x_adj_for_chain_team"] = np.where(
    chains_with_faceoff["prior_faceoff_x_adj_for_event_team"].notna(),
    np.where(
        same_team_faceoff,
        chains_with_faceoff["prior_faceoff_x_adj_for_event_team"],
        -chains_with_faceoff["prior_faceoff_x_adj_for_event_team"],
    ),
    np.nan,
)

chains_with_faceoff["prior_faceoff_y_adj_for_chain_team"] = np.where(
    chains_with_faceoff["prior_faceoff_y_adj_for_event_team"].notna(),
    np.where(
        same_team_faceoff,
        chains_with_faceoff["prior_faceoff_y_adj_for_event_team"],
        -chains_with_faceoff["prior_faceoff_y_adj_for_event_team"],
    ),
    np.nan,
)

chains_with_faceoff[
    [
        "chain_team_name",
        "prior_faceoff_team",
        "prior_faceoff_x_adj_for_event_team",
        "prior_faceoff_x_adj_for_chain_team",
        "prior_faceoff_y_adj_for_event_team",
        "prior_faceoff_y_adj_for_chain_team",
    ]
].head()

,chain_team_name,prior_faceoff_team,prior_faceoff_x_adj_for_event_team,prior_faceoff_x_adj_for_chain_team,prior_faceoff_y_adj_for_event_team,prior_faceoff_y_adj_for_chain_team
0,CLE,CLE,69.104380,69.104380,21.376137,21.376137
1,GR,CLE,-19.413269,19.413269,20.873200,-20.873200
2,GR,CLE,-19.413269,19.413269,20.873200,-20.873200
3,GR,CLE,-19.413269,19.413269,20.873200,-20.873200
4,GR,CLE,69.104380,-69.104380,20.873200,-20.873200


In [23]:
# Create offensive-zone faceoff context flags using a five-second window

CONTEXT_WINDOW_SECONDS = 5
OFFENSIVE_BLUE_LINE_X_ADJ = 25

chains_with_faceoff["has_prior_faceoff_in_sequence"] = (
    chains_with_faceoff["prior_faceoff_event_id"].notna()
)

chains_with_faceoff["prior_faceoff_within_5s"] = (
    chains_with_faceoff["seconds_since_prior_faceoff"].between(
        0,
        CONTEXT_WINDOW_SECONDS,
        inclusive="both",
    )
)

chains_with_faceoff["prior_faceoff_in_offensive_zone_for_chain"] = (
    chains_with_faceoff["prior_faceoff_x_adj_for_chain_team"] >= OFFENSIVE_BLUE_LINE_X_ADJ
)

chains_with_faceoff["oz_faceoff_context_5s"] = (
    chains_with_faceoff["is_offensive_zone_origin"]
    & chains_with_faceoff["prior_faceoff_within_5s"]
    & chains_with_faceoff["prior_faceoff_in_offensive_zone_for_chain"]
)

faceoff_context_validation = {
    "rows": len(chains_with_faceoff),
    "has_prior_faceoff_in_sequence": int(chains_with_faceoff["has_prior_faceoff_in_sequence"].sum()),
    "prior_faceoff_within_5s": int(chains_with_faceoff["prior_faceoff_within_5s"].sum()),
    "oz_faceoff_context_5s": int(chains_with_faceoff["oz_faceoff_context_5s"].sum()),
    "oz_faceoff_context_non_oz_origin": int(
        (
            chains_with_faceoff["oz_faceoff_context_5s"]
            & ~chains_with_faceoff["is_offensive_zone_origin"]
        ).sum()
    ),
    "duplicate_chain_ids": chains_with_faceoff.duplicated("chain_id").sum(),
}

faceoff_context_validation

{'rows': 48673,
 'has_prior_faceoff_in_sequence': 48652,
 'prior_faceoff_within_5s': 3808,
 'oz_faceoff_context_5s': 3615,
 'oz_faceoff_context_non_oz_origin': 0,
 'duplicate_chain_ids': np.int64(0)}

## 9. Add Sequence-Age Diagnostics

Compute how long the attacking team had been active in the current sequence before the origin shot.

This is retained as a diagnostic field, not as a primary context label.

A short sequence age can help flag quick attacks, but it is not equivalent to a true offensive-zone entry. The primary entry context will be built separately using an offensive-zone entry proxy.

In [24]:
# Build same-team event table by matching raw events to each chain's team within game, period, and sequence

same_team_event_context = event_context.merge(
    chains_with_faceoff[
        [
            "chain_id",
            "game_id",
            "period",
            "sequence_id",
            "chain_team_name",
        ]
    ],
    on=["game_id", "period", "sequence_id"],
    how="inner",
)

same_team_event_context = same_team_event_context[
    same_team_event_context["context_event_team"].eq(
        same_team_event_context["chain_team_name"]
    )
].copy()

same_team_event_context.shape

(2755727, 11)

In [25]:
# Compute first same-team event time in the sequence for each chain

same_team_sequence_start = (
    same_team_event_context
    .groupby("chain_id", as_index=False)
    .agg(
        first_same_team_event_time_in_sequence=("context_event_time", "min"),
        n_same_team_events_in_sequence=("context_event_id", "count"),
    )
)

chains_with_sequence_age = chains_with_faceoff.merge(
    same_team_sequence_start,
    on="chain_id",
    how="left",
    validate="one_to_one",
)

chains_with_sequence_age["seconds_since_first_same_team_event_in_sequence"] = (
    chains_with_sequence_age["anchor_origin_period_time"]
    - chains_with_sequence_age["first_same_team_event_time_in_sequence"]
)

chains_with_sequence_age[
    [
        "chain_id",
        "anchor_origin_period_time",
        "first_same_team_event_time_in_sequence",
        "seconds_since_first_same_team_event_in_sequence",
        "n_same_team_events_in_sequence",
    ]
].head()

,chain_id,anchor_origin_period_time,first_same_team_event_time_in_sequence,seconds_since_first_same_team_event_in_sequence,n_same_team_events_in_sequence
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s11_td...,591.730000,589.030000,2.700000,20
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,722.400000,620.030000,102.370000,89
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,736.330000,620.030000,116.300000,89
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,765.730000,620.030000,145.700000,89
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s13_t6...,923.800000,773.030000,150.770000,50


In [26]:
# Create sequence-age diagnostic flags

chains_with_sequence_age["early_same_team_sequence_5s_diagnostic"] = (
    chains_with_sequence_age["seconds_since_first_same_team_event_in_sequence"].between(
        0,
        CONTEXT_WINDOW_SECONDS,
        inclusive="both",
    )
)

chains_with_sequence_age["settled_sequence_gt5s_diagnostic"] = (
    chains_with_sequence_age["seconds_since_first_same_team_event_in_sequence"]
    > CONTEXT_WINDOW_SECONDS
)

sequence_age_validation = {
    "rows": len(chains_with_sequence_age),
    "missing_first_same_team_event_time": chains_with_sequence_age["first_same_team_event_time_in_sequence"].isna().sum(),
    "early_same_team_sequence_5s_diagnostic": int(chains_with_sequence_age["early_same_team_sequence_5s_diagnostic"].sum()),
    "settled_sequence_gt5s_diagnostic": int(chains_with_sequence_age["settled_sequence_gt5s_diagnostic"].sum()),
    "negative_sequence_age": int((chains_with_sequence_age["seconds_since_first_same_team_event_in_sequence"] < 0).sum()),
    "duplicate_chain_ids": chains_with_sequence_age.duplicated("chain_id").sum(),
}

sequence_age_validation

{'rows': 48673,
 'missing_first_same_team_event_time': np.int64(0),
 'early_same_team_sequence_5s_diagnostic': 3828,
 'settled_sequence_gt5s_diagnostic': 44843,
 'negative_sequence_age': 0,
 'duplicate_chain_ids': np.int64(0)}

## 10. Add Offensive-Zone Entry Proxy

Create a five-second offensive-zone entry proxy.

The ideal definition would use the exact moment the puck crosses the offensive blue line. This dataset does not provide continuous puck-crossing events, so this notebook approximates entry timing using the first same-team offensive-zone puck-movement event before the origin shot.

A chain is labeled as entry context when:

- the chain origin is in the offensive zone
- a same-team entry-candidate event occurred in the offensive zone
- that event occurred at or before the origin shot
- the origin shot occurred within 5 seconds of that event

This should be interpreted as an **entry proxy**, not a true rush label.

In [27]:
# Define event types that can plausibly represent offensive-zone entry or immediate post-entry puck movement

available_event_types = set(event_context["context_event_type"].dropna().unique())

entry_proxy_candidate_types = [
    event_type
    for event_type in [
        "controlledentry",
        "carry",
        "pass",
        "reception",
        "dumpin",
    ]
    if event_type in available_event_types
]

entry_proxy_candidate_types

['carry', 'pass', 'reception', 'dumpin']

In [28]:
# Build same-team event table with chain origin time attached

same_team_events_for_entry = same_team_event_context.merge(
    chains_with_sequence_age[
        [
            "chain_id",
            "anchor_origin_period_time",
            "is_offensive_zone_origin",
        ]
    ],
    on="chain_id",
    how="left",
    validate="many_to_one",
)

same_team_events_for_entry["context_event_x_adj_for_chain_team"] = (
    same_team_events_for_entry["context_event_x_adj_for_event_team"]
)

same_team_events_for_entry["context_event_y_adj_for_chain_team"] = (
    same_team_events_for_entry["context_event_y_adj_for_event_team"]
)

same_team_events_for_entry = same_team_events_for_entry[
    [
        "chain_id",
        "game_id",
        "period",
        "sequence_id",
        "anchor_origin_period_time",
        "is_offensive_zone_origin",
        "context_event_id",
        "context_event_time",
        "context_event_type",
        "context_event_x_adj_for_chain_team",
        "context_event_y_adj_for_chain_team",
    ]
].copy()

same_team_events_for_entry.head()

,chain_id,game_id,period,sequence_id,anchor_origin_period_time,is_offensive_zone_origin,context_event_id,context_event_time,context_event_type,context_event_x_adj_for_chain_team,context_event_y_adj_for_chain_team
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s2_td7...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2,125.270000,True,78,101.030000,faceoff,-69.204450,-21.876802
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s2_td7...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2,142.200000,True,78,101.030000,faceoff,-69.204450,-21.876802
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s2_td7...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2,155.270000,True,78,101.030000,faceoff,-69.204450,-21.876802
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s2_td7...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2,161.830000,True,78,101.030000,faceoff,-69.204450,-21.876802
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s2_td7...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,2,174.970000,True,78,101.030000,faceoff,-69.204450,-21.876802


In [29]:
# Identify same-team offensive-zone entry-candidate events that occurred at or before the origin shot

same_team_oz_entry_candidate_events = same_team_events_for_entry[
    same_team_events_for_entry["is_offensive_zone_origin"]
    & same_team_events_for_entry["context_event_type"].isin(entry_proxy_candidate_types)
    & same_team_events_for_entry["context_event_x_adj_for_chain_team"].ge(OFFENSIVE_BLUE_LINE_X_ADJ)
    & same_team_events_for_entry["context_event_time"].le(
        same_team_events_for_entry["anchor_origin_period_time"]
    )
].copy()

same_team_oz_entry_candidate_events = same_team_oz_entry_candidate_events.sort_values(
    ["chain_id", "context_event_time"]
)

same_team_oz_entry_candidate_events.head()

,chain_id,game_id,period,sequence_id,anchor_origin_period_time,is_offensive_zone_origin,context_event_id,context_event_time,context_event_type,context_event_x_adj_for_chain_team,context_event_y_adj_for_chain_team
568,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s11_td...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,11,591.730000,True,560,589.100000,pass,69.100876,23.385292
569,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s11_td...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,11,591.730000,True,561,589.800000,reception,61.053814,12.320587
570,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s11_td...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,11,591.730000,True,562,589.930000,pass,58.539110,16.847057
571,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s11_td...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,11,591.730000,True,563,591.000000,reception,29.368523,31.935295
601,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,00b0366a-95c6-5250-2dae-e3dd5c4198bc,1,12,722.400000,True,598,637.530000,carry,25.955010,-36.461760


In [30]:
# Select the first same-team offensive-zone entry-candidate event before the origin shot

first_entry_proxy_event_by_chain = (
    same_team_oz_entry_candidate_events
    .groupby("chain_id", as_index=False)
    .first()
    .rename(
        columns={
            "context_event_id": "entry_proxy_event_id",
            "context_event_time": "entry_proxy_time",
            "context_event_type": "entry_proxy_event_type",
            "context_event_x_adj_for_chain_team": "entry_proxy_x_adj_for_chain_team",
            "context_event_y_adj_for_chain_team": "entry_proxy_y_adj_for_chain_team",
        }
    )
)

first_entry_proxy_event_by_chain = first_entry_proxy_event_by_chain[
    [
        "chain_id",
        "entry_proxy_event_id",
        "entry_proxy_time",
        "entry_proxy_event_type",
        "entry_proxy_x_adj_for_chain_team",
        "entry_proxy_y_adj_for_chain_team",
    ]
]

first_entry_proxy_event_by_chain.head()

,chain_id,entry_proxy_event_id,entry_proxy_time,entry_proxy_event_type,entry_proxy_x_adj_for_chain_team,entry_proxy_y_adj_for_chain_team
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s11_td...,560,589.100000,pass,69.100876,23.385292
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,598,637.530000,carry,25.955010,-36.461760
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,598,637.530000,carry,25.955010,-36.461760
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,598,637.530000,carry,25.955010,-36.461760
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s13_t6...,808,849.400000,pass,50.096190,-39.982353


In [31]:
# Attach the entry proxy to each chain and calculate time from entry proxy to origin shot

chains_with_entry_proxy = chains_with_sequence_age.merge(
    first_entry_proxy_event_by_chain,
    on="chain_id",
    how="left",
    validate="one_to_one",
)

chains_with_entry_proxy["seconds_since_entry_proxy"] = (
    chains_with_entry_proxy["anchor_origin_period_time"]
    - chains_with_entry_proxy["entry_proxy_time"]
)

chains_with_entry_proxy[
    [
        "chain_id",
        "anchor_origin_period_time",
        "entry_proxy_time",
        "seconds_since_entry_proxy",
        "entry_proxy_event_type",
        "entry_proxy_x_adj_for_chain_team",
        "is_offensive_zone_origin",
    ]
].head()

,chain_id,anchor_origin_period_time,entry_proxy_time,seconds_since_entry_proxy,entry_proxy_event_type,entry_proxy_x_adj_for_chain_team,is_offensive_zone_origin
0,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s11_td...,591.730000,589.100000,2.630000,pass,69.100876,True
1,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,722.400000,637.530000,84.870000,carry,25.955010,True
2,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,736.330000,637.530000,98.800000,carry,25.955010,True
3,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s12_t6...,765.730000,637.530000,128.200000,carry,25.955010,True
4,00b0366a-95c6-5250-2dae-e3dd5c4198bc_p1_s13_t6...,923.800000,849.400000,74.400000,pass,50.096190,True


In [32]:
# Create five-second offensive-zone entry proxy flags

chains_with_entry_proxy["has_entry_proxy_in_sequence"] = (
    chains_with_entry_proxy["entry_proxy_event_id"].notna()
)

chains_with_entry_proxy["entry_proxy_within_5s"] = (
    chains_with_entry_proxy["seconds_since_entry_proxy"].between(
        0,
        CONTEXT_WINDOW_SECONDS,
        inclusive="both",
    )
)

chains_with_entry_proxy["oz_entry_context_5s_proxy"] = (
    chains_with_entry_proxy["is_offensive_zone_origin"]
    & chains_with_entry_proxy["entry_proxy_within_5s"]
)

entry_proxy_validation = {
    "rows": len(chains_with_entry_proxy),
    "entry_proxy_candidate_types": entry_proxy_candidate_types,
    "same_team_oz_entry_candidate_events": len(same_team_oz_entry_candidate_events),
    "has_entry_proxy_in_sequence": int(chains_with_entry_proxy["has_entry_proxy_in_sequence"].sum()),
    "entry_proxy_within_5s": int(chains_with_entry_proxy["entry_proxy_within_5s"].sum()),
    "oz_entry_context_5s_proxy": int(chains_with_entry_proxy["oz_entry_context_5s_proxy"].sum()),
    "negative_seconds_since_entry_proxy": int(
        (chains_with_entry_proxy["seconds_since_entry_proxy"] < 0).sum()
    ),
    "duplicate_chain_ids": chains_with_entry_proxy.duplicated("chain_id").sum(),
}

entry_proxy_validation

{'rows': 48673,
 'entry_proxy_candidate_types': ['carry', 'pass', 'reception', 'dumpin'],
 'same_team_oz_entry_candidate_events': 456931,
 'has_entry_proxy_in_sequence': 44616,
 'entry_proxy_within_5s': 11895,
 'oz_entry_context_5s_proxy': 11895,
 'negative_seconds_since_entry_proxy': 0,
 'duplicate_chain_ids': np.int64(0)}

## 11. Create Final Shot Context Labels

Create one mutually exclusive context label per chain.

Priority order:

1. offensive-zone faceoff within 5 seconds
2. offensive-zone entry proxy within 5 seconds
3. settled offensive-zone possession proxy
4. other / unknown

The priority order matters. A shot immediately after an offensive-zone faceoff may also be early in a sequence, but the more specific context is the faceoff.

In [33]:
# Create mutually exclusive shot-context labels

context_labeled = chains_with_entry_proxy.copy()

context_labeled["settled_offensive_zone_context_proxy"] = (
    context_labeled["is_offensive_zone_origin"]
    & context_labeled["settled_sequence_gt5s_diagnostic"]
    & ~context_labeled["oz_faceoff_context_5s"]
    & ~context_labeled["oz_entry_context_5s_proxy"]
)

context_labeled["shot_context"] = np.select(
    [
        context_labeled["oz_faceoff_context_5s"],
        context_labeled["oz_entry_context_5s_proxy"],
        context_labeled["settled_offensive_zone_context_proxy"],
    ],
    [
        "oz_faceoff_5s",
        "oz_entry_5s_proxy",
        "settled_offensive_zone_proxy",
    ],
    default="other_or_unknown",
)

context_labeled["shot_context"].value_counts(dropna=False)

shot_context
settled_offensive_zone_proxy    35038
oz_entry_5s_proxy                8894
oz_faceoff_5s                    3615
other_or_unknown                 1126
Name: count, dtype: int64

In [34]:
# Create a context-detail table for review

context_summary = (
    context_labeled
    .groupby("shot_context", observed=False)
    .agg(
        chains=("chain_id", "count"),
        is_5v5_rate=("is_5v5", "mean"),
        outside_origin_rate=("anchor_origin_location", lambda s: s.eq("outside").mean()),
        offensive_zone_origin_rate=("is_offensive_zone_origin", "mean"),
        tracking_available_rate=("origin_tracking_available", "mean"),
        tracking_error_rate=("origin_tracking_error_flag", "mean"),
        mean_seconds_since_faceoff=("seconds_since_prior_faceoff", "mean"),
        mean_seconds_since_entry_proxy=("seconds_since_entry_proxy", "mean"),
        mean_seconds_since_first_same_team_event=("seconds_since_first_same_team_event_in_sequence", "mean"),
    )
    .reset_index()
    .sort_values("chains", ascending=False)
)

context_summary

,shot_context,chains,is_5v5_rate,outside_origin_rate,offensive_zone_origin_rate,tracking_available_rate,tracking_error_rate,mean_seconds_since_faceoff,mean_seconds_since_entry_proxy,mean_seconds_since_first_same_team_event
3,settled_offensive_zone_proxy,35038,0.721017,0.618500,1.000000,0.933900,0.000514,71.981106,55.449739,71.981101
1,oz_entry_5s_proxy,8894,0.816955,0.562514,1.000000,0.938948,0.000337,32.014339,2.203970,32.013997
2,oz_faceoff_5s,3615,0.817427,0.777593,1.000000,0.889903,0.000277,2.927643,2.401519,2.927643
0,other_or_unknown,1126,0.775311,0.974245,0.041741,0.935169,0.000888,55.140443,NaN,54.208354


## 12. Context Waterfall For 5v5 Outside-Origin Offensive-Zone Analysis

Build the denominator table that Notebook 09 will use.

This checks how many chains remain after applying:

- 5v5
- outside-origin
- offensive-zone origin
- tracking available
- no tracking error flag

Then it breaks the final cohort down by context.

In [35]:
# Define the intended Notebook 09 analysis cohort

analysis_mask_09 = (
    context_labeled["is_5v5"]
    & context_labeled["anchor_origin_location"].eq("outside")
    & context_labeled["is_offensive_zone_origin"]
    & context_labeled["origin_tracking_available"]
    & ~context_labeled["origin_tracking_error_flag"]
)

analysis_09_candidate = context_labeled[analysis_mask_09].copy()

print("analysis_09_candidate:", analysis_09_candidate.shape)
analysis_09_candidate["shot_context"].value_counts(dropna=False)

analysis_09_candidate: (21536, 111)


shot_context
settled_offensive_zone_proxy    15387
oz_entry_5s_proxy                4020
oz_faceoff_5s                    2115
other_or_unknown                   14
Name: count, dtype: int64

In [36]:
# Build denominator waterfall for the future context-stratified analysis

context_waterfall = pd.DataFrame(
    [
        {
            "cohort": "all_origin_shot_chains",
            "chains": len(context_labeled),
        },
        {
            "cohort": "is_5v5",
            "chains": int(context_labeled["is_5v5"].sum()),
        },
        {
            "cohort": "is_5v5_outside_origin",
            "chains": int(
                (
                    context_labeled["is_5v5"]
                    & context_labeled["anchor_origin_location"].eq("outside")
                ).sum()
            ),
        },
        {
            "cohort": "is_5v5_outside_origin_offensive_zone",
            "chains": int(
                (
                    context_labeled["is_5v5"]
                    & context_labeled["anchor_origin_location"].eq("outside")
                    & context_labeled["is_offensive_zone_origin"]
                ).sum()
            ),
        },
        {
            "cohort": "is_5v5_outside_origin_offensive_zone_tracking_available",
            "chains": int(
                (
                    context_labeled["is_5v5"]
                    & context_labeled["anchor_origin_location"].eq("outside")
                    & context_labeled["is_offensive_zone_origin"]
                    & context_labeled["origin_tracking_available"]
                ).sum()
            ),
        },
        {
            "cohort": "is_5v5_outside_origin_offensive_zone_tracking_available_no_error_flag",
            "chains": int(analysis_mask_09.sum()),
        },
    ]
)

context_waterfall["pct_of_all"] = context_waterfall["chains"] / len(context_labeled)
context_waterfall["pct_of_previous"] = context_waterfall["chains"] / context_waterfall["chains"].shift(1)
context_waterfall.loc[0, "pct_of_previous"] = 1.0

context_waterfall

,cohort,chains,pct_of_all,pct_of_previous
0,all_origin_shot_chains,48673,1.000000,1.000000
1,is_5v5,36357,0.746964,0.746964
2,is_5v5_outside_origin,23754,0.488032,0.653354
3,is_5v5_outside_origin_offensive_zone,22919,0.470877,0.964848
4,is_5v5_outside_origin_offensive_zone_tracking_...,21549,0.442730,0.940224
5,is_5v5_outside_origin_offensive_zone_tracking_...,21536,0.442463,0.999397


In [37]:
# Break the intended analysis cohort down by context

analysis_context_counts = (
    analysis_09_candidate["shot_context"]
    .value_counts(dropna=False)
    .rename_axis("shot_context")
    .reset_index(name="chains")
)

analysis_context_counts["pct_of_analysis_candidate"] = (
    analysis_context_counts["chains"] / len(analysis_09_candidate)
)

analysis_context_counts

,shot_context,chains,pct_of_analysis_candidate
0,settled_offensive_zone_proxy,15387,0.714478
1,oz_entry_5s_proxy,4020,0.186664
2,oz_faceoff_5s,2115,0.098208
3,other_or_unknown,14,0.000650


## 13. Final Validation

Validate that the context-labeled dataset still has one row per chain and that the primary context fields are populated.

This notebook should preserve the chain dataset and only add context labels.

In [38]:
# Validate final context-labeled dataset

final_context_validation = {
    "rows": len(context_labeled),
    "unique_chain_ids": context_labeled["chain_id"].nunique(),
    "duplicate_chain_ids": context_labeled.duplicated("chain_id").sum(),
    "missing_shot_context": context_labeled["shot_context"].isna().sum(),
    "missing_origin_x_adj": context_labeled["anchor_origin_x_adj"].isna().sum(),
    "missing_chain_team_name": context_labeled["chain_team_name"].isna().sum(),
    "negative_seconds_since_entry_proxy": int((context_labeled["seconds_since_entry_proxy"] < 0).sum()),
    "negative_seconds_since_prior_faceoff": int((context_labeled["seconds_since_prior_faceoff"] < 0).sum()),
    "analysis_09_candidate_rows": len(analysis_09_candidate),
    "analysis_09_candidate_duplicate_chain_ids": analysis_09_candidate.duplicated("chain_id").sum(),
    "analysis_09_candidate_behind_blue_line_rows": int(
        analysis_09_candidate["is_behind_offensive_blue_line_origin"].sum()
    ),
}

final_context_validation

{'rows': 48673,
 'unique_chain_ids': 48673,
 'duplicate_chain_ids': np.int64(0),
 'missing_shot_context': np.int64(0),
 'missing_origin_x_adj': np.int64(2),
 'missing_chain_team_name': np.int64(0),
 'negative_seconds_since_entry_proxy': 0,
 'negative_seconds_since_prior_faceoff': 0,
 'analysis_09_candidate_rows': 21536,
 'analysis_09_candidate_duplicate_chain_ids': np.int64(0),
 'analysis_09_candidate_behind_blue_line_rows': 0}

## 14. Save Context-Labeled Dataset

Save the context-labeled origin-shot chain dataset for Notebook 09.

Output:

- `data/processed/origin_shot_sequences_context_labeled.parquet`

In [39]:
# Save the context-labeled origin-shot chain dataset

output_path = DATA_PROCESSED / "origin_shot_sequences_context_labeled.parquet"

context_labeled.to_parquet(output_path, index=False)

print("Saved:", output_path)
print("Shape:", context_labeled.shape)

Saved: c:\Users\rinal\hockey-analytics\outside-shot-value\data\processed\origin_shot_sequences_context_labeled.parquet
Shape: (48673, 111)


In [40]:
# Confirm the saved context-labeled dataset can be read back

saved_check = pd.read_parquet(DATA_PROCESSED / "origin_shot_sequences_context_labeled.parquet")

readback_validation = {
    "rows": len(saved_check),
    "unique_chain_ids": saved_check["chain_id"].nunique(),
    "duplicate_chain_ids": saved_check.duplicated("chain_id").sum(),
    "missing_shot_context": saved_check["shot_context"].isna().sum(),
    "shot_context_counts": saved_check["shot_context"].value_counts(dropna=False).to_dict(),
}

readback_validation

{'rows': 48673,
 'unique_chain_ids': 48673,
 'duplicate_chain_ids': np.int64(0),
 'missing_shot_context': np.int64(0),
 'shot_context_counts': {'settled_offensive_zone_proxy': 35038,
  'oz_entry_5s_proxy': 8894,
  'oz_faceoff_5s': 3615,
  'other_or_unknown': 1126}}